# 실험 02: Ragas (실무 RAG 평가의 사실상 표준)

## 1. 실험 제목과 목표
- **제목**: 내 RAG의 점수는 몇 점?
- **목표**: 실무에서 가장 많이 쓰이는 오픈소스 RAG 평가 라이브러리인 **Ragas**를 사용하여, 우리가 구축한 RAG의 답변 품질(근거성, 관련성 등)을 0.0 ~ 1.0 사이의 정량적 수치로 뽑아내는 방법을 배웁니다.

## 2. 사전 준비
> **주의**: 이 노트북을 실행하려면 `ragas` 패키지와 `datasets` 패키지가 필요합니다.
> 터미널에서 아래 명령어를 실행해 주세요.
> `pip install ragas datasets pandas`

## 3. 라이브러리 import

In [9]:
import os
import pandas as pd
from dotenv import load_dotenv

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

C:\Users\USER\AppData\Local\Temp\ipykernel_26588\3298728909.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\USER\AppData\Local\Temp\ipykernel_26588\3298728909.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\USER\AppData\Local\Temp\ipykernel_26588\3298728909.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\USER\AppData\Local\Temp\ipykernel_26588\

## 4. 환경 변수 로드 및 모델 준비

In [10]:
load_dotenv()

# Ragas의 내부 채점자로 활약할 LLM과 임베딩 모델입니다.
eval_llm = ChatOpenAI(model="gpt-4o-mini")
eval_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 5. Ragas 규격에 맞는 데이터셋(Dataset) 준비
Ragas는 기본적으로 `[question(질문), answer(봇의 답변), contexts(검색된 문서 목록), ground_truth(실제 정답)]` 4가지 쌍을 요구합니다.

In [11]:
# 실무에서는 엑셀이나 CSV에 있는 수백 개의 평가 셋을 불러와서 사용하지만, 
# 여기서는 2개의 케이스를 하드코딩해서 테스트합니다.

data_samples = {
    "question": [
        "프랑스의 수도는 어디야?", 
        "아이폰 15의 출시일은 언제야?"
    ],
    "answer": [
        "프랑스의 수도는 파리입니다.", # 아주 훌륭한 답변
        "아이폰 15는 2020년에 출시되었습니다." # 할루시네이션(거짓말) 답변
    ],
    "contexts": [
        ["파리는 프랑스의 수도이자 가장 큰 도시입니다.", "프랑스는 유럽에 있습니다."], 
        ["애플은 2023년 9월에 아이폰 15 시리즈를 전격 공개했습니다."]
    ],
    "ground_truth": [
        "파리", 
        "2023년 9월"
    ]
}

# 파이썬 딕셔너리를 HuggingFace Dataset 객체로 변환합니다. (Ragas가 요구하는 포맷)
dataset = Dataset.from_dict(data_samples)

In [12]:
dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})

## 6. Ragas 평가 실행
교안에 명시된 평가 항목 중 가장 핵심적인 두 가지 지표를 뽑아봅니다.
1. **Faithfulness (근거성)**: 봇의 답변이 검색된 Context(문서)에 기반하고 있는가? (할루시네이션 검사)
2. **Answer Relevancy (답변 관련성)**: 봇의 답변이 사용자의 질문 의도를 정확히 해결했는가? (동문서답 검사)

In [13]:
print("=== [Ragas 평가 시작] ===")
print("여러 개의 평가를 병렬로 진행하므로 수십 초 정도 소요될 수 있습니다...")

# Ragas evaluate 함수 실행
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=eval_llm, # 채점을 진행할 LLM 지정
    embeddings=eval_embeddings # 텍스트 유사도 검사에 쓸 임베딩 모델 지정
)

print("\n=== [평가 완료] ===")
print("전체 평균 점수:")
print(result)

=== [Ragas 평가 시작] ===
여러 개의 평가를 병렬로 진행하므로 수십 초 정도 소요될 수 있습니다...


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]


=== [평가 완료] ===
전체 평균 점수:
{'faithfulness': 0.5000, 'answer_relevancy': 0.5551}


## 7. 결과를 데이터프레임(표)으로 예쁘게 보기

In [16]:
# 결과를 Pandas DataFrame으로 변환하여 각 행(질문)별 성적표를 확인합니다.
df = result.to_pandas()

print("\n=== [문항별 상세 성적표] ===")
df.head()


=== [문항별 상세 성적표] ===


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,프랑스의 수도는 어디야?,"[파리는 프랑스의 수도이자 가장 큰 도시입니다., 프랑스는 유럽에 있습니다.]",프랑스의 수도는 파리입니다.,파리,1.0,0.338015
1,아이폰 15의 출시일은 언제야?,[애플은 2023년 9월에 아이폰 15 시리즈를 전격 공개했습니다.],아이폰 15는 2020년에 출시되었습니다.,2023년 9월,0.0,0.772275


## 8. 결과 해석

1. **프랑스 수도 질문 (인덱스 0)**:
   - 답변이 완벽하고 문서 내용과 일치하므로 `faithfulness`와 `answer_relevancy` 모두 1.0(만점)에 가까운 수치가 나옵니다.
2. **아이폰 출시일 질문 (인덱스 1)**:
   - 문서는 "2023년 9월"이라고 했는데 봇이 "2020년"이라고 대답했으므로, **근거성(faithfulness)이 0.0점(낙제)**으로 떨어집니다.
   - 하지만 "출시일은 2020년입니다"라는 문장 형태 자체는 '출시일이 언제야?'라는 질문의 형태에 답하고 있으므로, **답변 관련성(answer_relevancy)**은 어느 정도 점수가 나올 수 있습니다.

실무에서는 모델이나 RAG 알고리즘을 변경할 때마다 이 평가 코드를 돌려서, **"우리 팀의 Faithfulness 평균이 0.85에서 0.92로 올랐다!"**라고 정량적으로 보고하는 것이 핵심입니다.